# Binary search — study environment

Work top to bottom. Implement the stubs, run the check cells, then compare with `solution.py` only after you are stuck.

**Invariant to keep saying out loud:** after every iteration, the answer (if it exists) still lives in `nums[lo:hi]`.

Convention used here: half-open interval `[lo, hi)` — `lo` is in, `hi` is out. Loop while `lo < hi`. Never read `nums[hi]`.

In [ ]:
from __future__ import annotations


def trace_search(nums: list[int], target: int) -> int:
    """Classic search that prints lo, mid, hi each step. Study this; do not copy blindly."""
    lo, hi = 0, len(nums)
    step = 0
    print(f"nums={nums}  target={target}")
    while lo < hi:
        mid = lo + (hi - lo) // 2
        print(f"  step {step}: lo={lo} mid={mid} hi={hi}  nums[mid]={nums[mid]}")
        step += 1
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    print(f"  miss: lo=hi={lo}")
    return -1


trace_search([1, 3, 5, 7, 9, 11], 7)
print("---")
trace_search([1, 3, 5, 7, 9, 11], 6)

## Why this works

| Move | New interval | Why it is safe |
|---|---|---|
| `nums[mid] < target` | `[mid + 1, hi)` | `mid` is too small, discard it |
| `nums[mid] > target` | `[lo, mid)` | `mid` is too big, discard it |
| `nums[mid] == target` | done | found |

`mid = lo + (hi - lo) // 2` always lands in `[lo, hi)` when `lo < hi`, so you never infinite-loop on `mid == lo` with a shrinking interval.

Python `int` does not overflow; the `lo + (hi - lo) // 2` form is still the one interviewers expect you to mention (C/Java `int` overflow).

## 1. Classic search

Return **any** index of `target`, else `-1`. Array is sorted non-decreasing.

In [ ]:
def binary_search(nums: list[int], target: int) -> int:
    raise NotImplementedError


def _check_binary_search():
    cases = [
        ([1, 3, 5, 7, 9], 5, {2}),
        ([1, 3, 5, 7, 9], 1, {0}),
        ([1, 3, 5, 7, 9], 9, {4}),
        ([1, 3, 5, 7, 9], 4, {-1}),
        ([], 1, {-1}),
        ([7], 7, {0}),
        ([2, 2, 2], 2, {0, 1, 2}),
    ]
    for nums, target, expected in cases:
        got = binary_search(nums, target)
        assert got in expected, (nums, target, got, expected)
    print("binary_search: ok")


_check_binary_search()

## 2. Lower bound (first `>= target`)

This is the template you should default to. It always returns an **insertion index** in `[0, n]`.

- `nums[i] >= target` for the first time at `i = lower_bound(...)`
- Same as Python `bisect.bisect_left`
- First occurrence of `target` is this index **if** `i < n and nums[i] == target`

In [ ]:
import bisect


def lower_bound(nums: list[int], target: int) -> int:
    raise NotImplementedError


def _check_lower_bound():
    arrays = [
        [],
        [1],
        [1, 2, 2, 2, 5],
        list(range(0, 40, 2)),
    ]
    targets = [-1, 0, 1, 2, 3, 5, 6, 17, 18, 100]
    for nums in arrays:
        for target in targets:
            got = lower_bound(nums, target)
            want = bisect.bisect_left(nums, target)
            assert got == want, (nums, target, got, want)
    print("lower_bound: ok")


_check_lower_bound()

## 3. Upper bound (first `> target`)

Same loop, different predicate: shrink right when `nums[mid] > target`, otherwise discard `mid` to the left (`lo = mid + 1`).

Last occurrence of `target` is `upper_bound(...) - 1` if that index is in range and equals `target`.

In [ ]:
def upper_bound(nums: list[int], target: int) -> int:
    raise NotImplementedError


def _check_upper_bound():
    arrays = [[], [1], [1, 2, 2, 2, 5], list(range(0, 40, 2))]
    targets = [-1, 0, 1, 2, 3, 5, 6, 17, 18, 100]
    for nums in arrays:
        for target in targets:
            got = upper_bound(nums, target)
            want = bisect.bisect_right(nums, target)
            assert got == want, (nums, target, got, want)
    print("upper_bound: ok")


_check_upper_bound()

## 4. Search on a predicate (`first True`)

Most "binary search on the answer" problems are this: there is a monotonic boolean `f(x)` that is `False False ... True True`. Find the smallest `x` in `[lo, hi)` with `f(x)` true, or return `hi`.

Example: integer square root — largest `x` with `x * x <= n` is `first_true(lambda x: x * x > n, 0, n + 1) - 1`.

In [ ]:
from collections.abc import Callable


def first_true(pred: Callable[[int], bool], lo: int, hi: int) -> int:
    """Smallest x in [lo, hi) with pred(x), or hi if none."""
    raise NotImplementedError


def isqrt(n: int) -> int:
    """Largest integer x with x * x <= n. Assume n >= 0."""
    raise NotImplementedError


def _check_first_true_and_isqrt():
    flags = [False, False, False, True, True]
    assert first_true(lambda i: flags[i], 0, 5) == 3
    assert first_true(lambda i: False, 0, 5) == 5
    assert first_true(lambda i: True, 0, 5) == 0
    for n in range(0, 50):
        x = isqrt(n)
        assert x * x <= n < (x + 1) * (x + 1)
    print("first_true / isqrt: ok")


_check_first_true_and_isqrt()

## 5. Rotated sorted array (distinct values)

`[4, 5, 6, 7, 0, 1, 2]` is a sorted array rotated at some pivot.

At `mid`, **one** of the two halves `[lo, mid]` or `[mid, hi)` is fully sorted. Check whether `target` sits in the sorted half; discard the other.

Watch the endpoints: `nums[lo] <= target < nums[mid]` vs `nums[mid] < target <= nums[hi - 1]`.

In [ ]:
def search_rotated(nums: list[int], target: int) -> int:
    raise NotImplementedError


def _check_search_rotated():
    cases = [
        ([4, 5, 6, 7, 0, 1, 2], 0, 4),
        ([4, 5, 6, 7, 0, 1, 2], 3, -1),
        ([1], 0, -1),
        ([1], 1, 0),
        ([3, 1], 1, 1),
        ([3, 1], 3, 0),
        ([6, 7, 1, 2, 3, 4, 5], 7, 1),
        ([6, 7, 1, 2, 3, 4, 5], 5, 6),
    ]
    for nums, target, expected in cases:
        got = search_rotated(nums, target)
        assert got == expected, (nums, target, got, expected)
    print("search_rotated: ok")


_check_search_rotated()

## Interview checklist

1. Is the sequence sorted, or is some **predicate** monotonic?
2. Empty input and single-element input.
3. Duplicates: do they want any / first / last / count (`upper - lower`)?
4. Return index vs insertion point vs boolean.
5. Overflow-safe `mid`; loop condition matches the interval (`lo < hi` vs `lo <= hi`).
6. Complexity: `O(log n)` comparisons, `O(1)` extra memory.

When every cell prints `ok`, copy the same functions into `workspace.py` and run:

```bash
pytest tests/algorithms/binary_search -q
```